In [ ]:
import pandas as pd
import io, zipfile, warnings
warnings.filterwarnings('ignore')
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
import numpy as np


In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

Saving Original Dataset.csv to Original Dataset.csv


In [ ]:
df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='latin1')
df.head(5)


,Name,Age,Gender,TO,TH,AH,BH,OX2,OXK,OX9,A,M,Rickettsia_Suspect,Acute_typhoid,Paratyphoid_A,Paratyphoid_B,Typhoid
0,gAAAAABogxzS5EEbf73hk5aCmUTW-sZm7JguQy991RjGTP...,5y,Male,"""1:80""","""1:80""","""1:80""","""1:80""","""1:320""","""1:320""","""1:80""","""1:80""","""1:80""",Yes,No,No,No,Negative
1,gAAAAABogxzSPxqjjXarnFAQwWiRBMIfqcWILZNj1Kco4t...,3.5y,Male,"""1:160""","""1:80""","""1:80""","""1:80""",NaN,NaN,NaN,NaN,NaN,NaN,Yes,No,No,Minimal
2,gAAAAABogxzSXhet-9e_mniQeaztJLKicAOe-sBhQhKt-G...,45y,Male,"""1:80""","""1:80""","""1:80""","""1:80""","""1:160""","""1:160""","""1:80""",NaN,NaN,Yes,No,No,No,Negative
3,gAAAAABogxzSEeL63vvluPQgs7voTJeL-H3Aus-MMeO-X4...,13y,Female,"""1:80""","""1:160""","""1:80""","""1:80""","""1:160""","""1:320""","""1:80""","""1:80""","""1:160""",Yes,No,No,No,Minimal
4,gAAAAABogxzSqjG6cpo0dqBciuhJf4axeyfgArg5dLYZAh...,12y,Female,"""1:160""","""1:320""","""1:80""","""1:80""","""1:160""","""1:160""","""1:320""","""1:80""","""1:80""",Yes,Yes,No,No,Positive


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1106 entries, 0 to 1105
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Name                1106 non-null   object
 1   Age                 1106 non-null   object
 2   Gender              1106 non-null   object
 3   TO                  1100 non-null   object
 4   TH                  1100 non-null   object
 5   AH                  1100 non-null   object
 6   BH                  1100 non-null   object
 7   OX2                 709 non-null    object
 8   OXK                 709 non-null    object
 9   OX9                 709 non-null    object
 10  A                   655 non-null    object
 11  M                   655 non-null    object
 12  Rickettsia_Suspect  709 non-null    object
 13  Acute_typhoid       1100 non-null   object
 14  Paratyphoid_A       1100 non-null   object
 15  Paratyphoid_B       1100 non-null   object
 16  Typhoid             1100

In [ ]:
df.columns = df.columns.str.strip()
data = df.copy()
for col in ['Name','Encrypted_Name']:
    if col in data.columns:
        data.drop(columns=[col], inplace=True)

le = LabelEncoder()
for col in data.columns:
    if data[col].dtype == 'object':
        data[col] = le.fit_transform(data[col].astype(str))

target = 'Acute_typhoid'
drop_cols = [c for c in ['Typhoid','Paratyphoid_A','Paratyphoid_B','Rickettsia_Suspect'] if c in data.columns]
X = data.drop(columns=[target] + drop_cols)
y = data[target]

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


# Step 1: Base Models (SVM, Naive Bayes, Decision Tree)
svm = SVC(probability=True, random_state=42)
gnb = GaussianNB()
dt  = DecisionTreeClassifier(random_state=42)

svm.fit(X_train_sc, y_train)
gnb.fit(X_train_sc, y_train)
dt.fit(X_train_sc, y_train)


# Step 2: Base model predictions একসাথে করা
meta_train = np.column_stack([
    svm.predict(X_train_sc),
    gnb.predict(X_train_sc),
    dt.predict(X_train_sc)
])


meta_test = np.column_stack([
    svm.predict(X_test_sc),
    gnb.predict(X_test_sc),
    dt.predict(X_test_sc)
])



# Step 3: LightGBM Metamodel
meta_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
meta_model.fit(meta_train, y_train)
y_pred = meta_model.predict(meta_test)

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f' Model    : Metamodel')
print(f' Accuracy : {acc*100:.2f}%')

 Model    : Metamodel
 Accuracy : 98.65%
